# Signature Flatbreads — Maintenance KPI Dashboard

**Lines:** P1 (Pancake) · T1 / T2 (Tortilla wrap) · L8 / L9

Calculates Target vs Actual, weighted KPI scores, and monthly bonus payouts (V2 scheme).


In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd()
if (ROOT / "kpi_calculator.py").exists():
    sys.path.insert(0, str(ROOT))
elif (ROOT / "signature_flatbread_kpi" / "kpi_calculator.py").exists():
    ROOT = ROOT / "signature_flatbread_kpi"
    sys.path.insert(0, str(ROOT))

from kpi_calculator import (
    load_config,
    targets_table,
    calculate_monthly_kpis,
    bonus_payouts,
    export_excel,
)

config = load_config(ROOT / "config" / "kpi_scheme.json")
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 140)
print("Scheme:", config["scheme_name"])
print("Lines:", ", ".join(f"{l['code']} ({l['name']})" for l in config["lines"]))


## 1. Targets (Year 1–3 + world class)

In [ ]:
targets = targets_table(config)
targets

## 2. Load actuals

Use the sample file to see scoring, then switch to your real monthly CSV (copied from the template).


In [ ]:
ACTUALS = ROOT / "data" / "monthly_actuals_SAMPLE.csv"
# ACTUALS = ROOT / "data" / "monthly_actuals_2026-07.csv"  # <-- your real file

raw = pd.read_csv(ACTUALS)
raw

## 3. Score each line (Actual vs Target)

In [ ]:
scored, gates = calculate_monthly_kpis(ACTUALS, config)

line_view = scored[[
    "line", "line_name",
    "oee_actual_pct", "oee_target_pct", "oee_score_pct", "oee_weighted",
    "breakdown_frequency_actual", "breakdown_frequency_target", "breakdown_frequency_weighted",
    "breakdown_hours_actual", "breakdown_hours_target", "breakdown_hours_weighted",
    "pm_compliance_actual_pct", "pm_compliance_target_pct", "pm_compliance_weighted",
    "mttr_actual_hrs", "mttr_target_hrs", "mttr_weighted",
    "mtbf_actual_hrs", "mtbf_target_hrs", "mtbf_weighted",
    "safety_lti_actual", "safety_weighted",
    "line_kpi_score_pct",
]]
line_view

## 4. Plant summary & gatekeepers

In [ ]:
print("Plant KPI score (avg of lines, before gates):", gates["plant_kpi_score_before_gates_pct"], "%")
print("Final bonus score %:", gates["final_bonus_score_pct"])
print("Gatekeeper notes:", gates["gatekeeper_notes"])

summary_chart = scored.set_index("line")["line_kpi_score_pct"]
ax = summary_chart.plot(kind="bar", title="Line KPI Score %", ylabel="Score %", ylim=(0, 100), figsize=(8, 4))
ax.axhline(gates["final_bonus_score_pct"], color="crimson", linestyle="--", label="Final bonus score")
ax.legend()


## 5. Bonus payouts by role

In [ ]:
payouts = bonus_payouts(gates["final_bonus_score_pct"], config)
payouts

## 6. Export Excel report for your manager

In [ ]:
out = ROOT / "output" / "KPI_Report.xlsx"
export_excel(scored, gates, payouts, out, config)
print("Saved:", out.resolve())


## Monthly data checklist

1. OEE inputs from production Excel for **P1, T1, T2, L8, L9**
2. Breakdown start/end from Teams + Excel → count + hours
3. PM jobs from Maintainer → planned vs completed on time
4. Safety LTI / EPW / audit colour from H&S
5. Paste into `monthly_actuals_YYYY-MM.csv` and re-run

Scores and overall bonus are capped at **100%**. Team performance drives the payout.
